# PS3: Address Geocoder (Perfect Pipeline)

This geocoder uses filtered GPS telemetry to calculate true centroids and mathematically computes the confidence radius from the GPS point standard deviations, based on our EDA findings regarding field visit integrity.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import os
import re

## 2. Data Loading

In [ ]:
base_path = r"../../Dataset/data"

In [ ]:
addresses = pd.read_csv(os.path.join(base_path, "shared", "addresses.csv"))
baseline = pd.read_csv(os.path.join(base_path, "ps3_geocoder", "baseline_geocodes.csv"))
field_visits = pd.read_csv(os.path.join(base_path, "shared", "field_visits.csv"))
surveyed = pd.read_csv(os.path.join(base_path, "ps3_geocoder", "surveyed_addresses.csv"))

## 3. High-Integrity Filter & Confidence Mathematics

In [ ]:
# As per EDA, only certain outcomes are valid and accuracy must be < 30m
valid_outcomes = ['met_borrower', 'met_family', 'cash_collected', 'door_locked']
valid_visits = field_visits[
    (field_visits['outcome'].isin(valid_outcomes)) &
    (field_visits['gps_accuracy_m'] <= 25) & 
    (field_visits['dwell_s'] >= 60)
]

In [ ]:
def calculate_centroid_and_confidence(df):
    weights = 1.0 / (df['gps_accuracy_m'] + 1e-5)
    mean_x = np.average(df['checkin_x'], weights=weights)
    mean_y = np.average(df['checkin_y'], weights=weights)
    var_x = np.average((df['checkin_x'] - mean_x)**2, weights=weights)
    var_y = np.average((df['checkin_y'] - mean_y)**2, weights=weights)
    radius = np.sqrt(var_x + var_y) + df['gps_accuracy_m'].mean()
    return pd.Series({'visit_x': mean_x, 'visit_y': mean_y, 'conf_radius': radius})

In [ ]:
address_gps = valid_visits.groupby('address_id').apply(calculate_centroid_and_confidence).reset_index()

In [ ]:
data = pd.merge(addresses, baseline, on='address_id', how='left')
data = pd.merge(data, address_gps, on='address_id', how='left')

In [ ]:
data['pred_x'] = data['visit_x'].fillna(data['geocoder_x'])
data['pred_y'] = data['visit_y'].fillna(data['geocoder_y'])

## 4. Evaluation

In [ ]:
eval_data = pd.merge(surveyed, data, on='address_id', how='inner')

In [ ]:
eval_data['model_error'] = np.sqrt((eval_data['pred_x'] - eval_data['surveyed_x'])**2 + (eval_data['pred_y'] - eval_data['surveyed_y'])**2)

In [ ]:
print("Advanced Model Metrics (m):")
print(f"Median Error: {eval_data['model_error'].median():.2f}")
print(f"90th Percentile: {eval_data['model_error'].quantile(0.90):.2f}")